# LLM-Driven Bayesian Modeling Tool for Reliability & Predictive Maintenance
### RAMS 2027 Tutorial — Kirtis Christensen

**Using Hubbard's Applied Information Economics (AIE) + Large Language Models**

---

## How to Use This Notebook

Run cells **top to bottom**. Each section corresponds to one AIE step.  
At each `## YOUR TURN` block, substitute your own problem description or data.

**What you will produce by the end:**
- A structured JSON decomposition of your reliability problem
- Value of Information (VoI) ranking of your measurements
- A calibrated Bayesian model with prior → likelihood → posterior
- A Monte Carlo simulation and decision recommendation

---

### Tutorial Sections

| # | Section | AIE Step |
|---|---------|----------|
| 1 | Environment Setup | — |
| 2 | Background: AIE + Bayesian Concepts | Framing |
| 3 | Step 1 — Define the Decision & Uncertainty | AIE Step 1 |
| 4 | Step 2 — LLM Problem Parser → JSON | AIE Step 2 |
| 5 | Step 3 — Value of Information (VoI) Ranking | AIE Step 3 |
| 6 | Step 4 — Calibrated Priors (Two Paths) | AIE Step 4 |
| 7 | Step 5 — Bayesian Update (Likelihood + Posterior) | AIE Step 5 |
| 8 | Step 6 — Chained Bayesian DAG Visualization | AIE Step 5 |
| 9 | Step 7 — Monte Carlo Simulation & Decision | AIE Step 5 |
|10 | Your Turn — Participant Exercise | All Steps |

---
## Section 1 — Environment Setup

Run this cell once to install required packages.  
On **Google Colab**: this works immediately with no local configuration.

In [ ]:
# Install required packages (safe to re-run)
import subprocess, sys

packages = [
    "numpy", "scipy", "pandas", "matplotlib", "seaborn",
    "networkx",       # DAG visualization
    "openai",         # LLM API (OpenAI-compatible; swap for any provider)
    "pymc",           # Bayesian inference
    "arviz",          # Posterior diagnostics
    "ipywidgets",     # Interactive sliders
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages ready.")

In [ ]:
# Core imports
import os, json, textwrap
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import pymc as pm
import arviz as az
import ipywidgets as widgets
from IPython.display import display, Markdown, JSON

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")
print("Environment ready.")

---
## Section 2 — Background: AIE + Bayesian Concepts

### Doug Hubbard's Applied Information Economics (AIE) — 5 Steps

| Step | Action | Bayesian Equivalent |
|------|--------|---------------------|
| 1 | Define the decision and what is uncertain | Identify random variables |
| 2 | Determine what measurements matter | VoI analysis |
| 3 | Measure highest-value items | Collect evidence / data |
| 4 | Use information to update beliefs | Bayesian update: prior × likelihood → posterior |
| 5 | Make the decision | Expected utility / loss function |

### The Two Persistent Barriers

**Barrier 1 — Distribution Quality**  
Engineers default to uniform or normal distributions when the physics, failure modes, and data suggest something very different (Weibull, Beta, Lognormal, etc.).

**Barrier 2 — Parameter Mapping**  
Even engineers who understand Bayes struggle to map their specific problem into:  
$$P(\theta \mid \text{data}) \propto P(\text{data} \mid \theta) \cdot P(\theta)$$

**This tool solves both** by using an LLM to parse the problem description, select the appropriate distributions, and generate the full JSON model structure automatically.

### Worked Example Throughout This Tutorial

> **Problem**: A centrifugal pump at a water treatment facility has experienced 3 failures in 24 months of operation.  
> Maintenance cost per intervention: \$4,200. Unplanned failure cost: \$18,000.  
> **Decision**: Should we implement a 6-month preventive maintenance (PM) schedule, or continue run-to-failure?

---
## Section 3 — Step 1: Define the Decision & Uncertainties

Before calling the LLM, we manually frame the decision.  
This is the **most important step** — a precise framing produces a precise model.

### Template: Decision Frame

In [ ]:
# ── DECISION FRAME ─────────────────────────────────────────────────────────────
# Fill in this dictionary for your problem. The LLM will use it in Section 4.

decision_frame = {
    "problem_description": """
        A centrifugal pump at a water treatment facility has experienced 3 failures
        in 24 months of operation. Each unplanned failure costs $18,000 in repairs
        and lost production. A preventive maintenance (PM) intervention costs $4,200.
        Historical industry data suggests mean time between failures (MTBF) for this
        pump class is 9–14 months under similar conditions, but our data is limited.
        We want to decide whether to implement a 6-month PM schedule or continue
        run-to-failure maintenance for the next 12 months.
    """,
    "decision_options": [
        "Implement 6-month preventive maintenance schedule",
        "Continue run-to-failure (reactive maintenance only)"
    ],
    "objective": "Minimize expected total maintenance cost over 12 months",
    "time_horizon_months": 12,
    "cost_pm_per_intervention_usd": 4200,
    "cost_unplanned_failure_usd": 18000,
    "observed_failures": 3,
    "observation_period_months": 24,
    "industry_mtbf_range_months": [9, 14],
}

display(Markdown("**Decision Frame Captured:**"))
display(Markdown(f"- **Decision**: {decision_frame['objective']}"))
display(Markdown(f"- **Options**: {decision_frame['decision_options']}"))
display(Markdown(f"- **Observed data**: {decision_frame['observed_failures']} failures in {decision_frame['observation_period_months']} months"))
display(Markdown(f"- **Costs**: PM=${decision_frame['cost_pm_per_intervention_usd']:,} | Failure=${decision_frame['cost_unplanned_failure_usd']:,}"))

---
## Section 4 — Step 2: LLM Problem Parser → Structured JSON

We send the problem description to an LLM and receive a **structured JSON** that:
- Identifies all uncertain quantities (the measurement nodes)
- Assigns appropriate probability distribution families to each
- Specifies whether each node is a **prior**, **likelihood**, or **posterior**
- Explains *why* each distribution was chosen

### Configure Your LLM API Key

> **Note**: The cell below uses the OpenAI API. You can substitute any OpenAI-compatible endpoint  
> (Azure OpenAI, Groq, Ollama, xAI Grok, etc.) by changing `base_url` and `model`.

In [ ]:
# ── LLM CONFIGURATION ──────────────────────────────────────────────────────────
# Option A: Set key directly (for local use only — never commit to git)
# Option B: Set environment variable OPENAI_API_KEY before launching Jupyter
# Option C: On Colab, use: from google.colab import userdata

from openai import OpenAI

# ── Edit these three lines for your provider ───────────────────────────────────
API_KEY   = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
BASE_URL  = "https://api.openai.com/v1"          # Change for other providers
MODEL     = "gpt-4o"                              # Or gpt-4-turbo, grok-3, etc.
# ───────────────────────────────────────────────────────────────────────────────

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print(f"LLM configured: {MODEL} @ {BASE_URL}")

In [ ]:
# ── SYSTEM PROMPT — Bayesian Model Engineer ────────────────────────────────────
SYSTEM_PROMPT = """
You are an expert Bayesian reliability engineer and statistician.
Given a natural-language reliability problem description, you must:

1. Identify all uncertain quantities (measurement nodes).
2. For each node, specify:
   - name: short identifier (snake_case)
   - description: plain-English explanation of what it represents
   - role: one of ["prior", "likelihood", "posterior", "decision_variable", "cost_node"]
   - distribution_family: e.g. Gamma, Weibull, Beta, Normal, Poisson, Exponential
   - distribution_parameters: dict of parameter names and initial values or ranges
   - justification: WHY this distribution family is correct for this node
   - data_source: "expert_estimate", "observed_data", or "derived"
   - depends_on: list of node names this node depends on (empty list if root node)
3. Identify the key Bayesian update chain as an ordered list of node names.
4. Return ONLY valid JSON. No markdown fences, no explanation outside the JSON.

JSON schema:
{
  "problem_summary": "string",
  "decision": "string",
  "measurement_nodes": [ { ...node fields... } ],
  "bayesian_update_chain": ["node_name", ...],
  "voi_candidates": ["node_name", ...],
  "model_notes": "string"
}
"""

def parse_problem_to_json(problem_text: str) -> dict:
    """Send problem description to LLM and return structured Bayesian JSON."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": problem_text}
        ],
        temperature=0.1,   # Low temperature for deterministic structured output
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

print("LLM parser function defined.")

In [ ]:
# ── CALL THE LLM ───────────────────────────────────────────────────────────────
# This sends the problem description and returns the structured Bayesian model.

bayesian_model_json = parse_problem_to_json(decision_frame["problem_description"])

# Pretty-print the result
print(json.dumps(bayesian_model_json, indent=2))

In [ ]:
# ── OFFLINE FALLBACK ───────────────────────────────────────────────────────────
# If you do not have an API key, use this pre-generated JSON for the pump example.
# Comment out the cell above and run this one instead.

OFFLINE_JSON = {
  "problem_summary": "Centrifugal pump with 3 failures in 24 months. Decide between 6-month PM schedule and run-to-failure over a 12-month horizon.",
  "decision": "Minimize expected total maintenance cost over 12 months",
  "measurement_nodes": [
    {
      "name": "failure_rate_lambda",
      "description": "True underlying failure rate of the pump (failures per month)",
      "role": "prior",
      "distribution_family": "Gamma",
      "distribution_parameters": {"alpha": 3.5, "beta": 12.0},
      "justification": "Gamma is the conjugate prior for a Poisson failure process. Alpha/beta seeded from industry MTBF range of 9-14 months (lambda ~ 0.07-0.11 failures/month).",
      "data_source": "expert_estimate",
      "depends_on": []
    },
    {
      "name": "observed_failures",
      "description": "Observed failure count: 3 failures in 24 months — the evidence term",
      "role": "likelihood",
      "distribution_family": "Poisson",
      "distribution_parameters": {"mu": "failure_rate_lambda * 24"},
      "justification": "Failures are discrete events; Poisson likelihood is standard for count data when events are independent and the rate is constant.",
      "data_source": "observed_data",
      "depends_on": ["failure_rate_lambda"]
    },
    {
      "name": "posterior_failure_rate",
      "description": "Updated belief about the true failure rate after observing 3 failures in 24 months",
      "role": "posterior",
      "distribution_family": "Gamma",
      "distribution_parameters": {"alpha": 6.5, "beta": 36.0},
      "justification": "Gamma-Poisson conjugacy: posterior Gamma(alpha + k, beta + t) = Gamma(3.5+3, 12+24) = Gamma(6.5, 36). Analytically exact.",
      "data_source": "derived",
      "depends_on": ["failure_rate_lambda", "observed_failures"]
    },
    {
      "name": "failures_next_12mo_pm",
      "description": "Predicted failures in next 12 months under PM schedule (reduced effective exposure)",
      "role": "decision_variable",
      "distribution_family": "Poisson",
      "distribution_parameters": {"mu": "posterior_failure_rate * 6"},
      "justification": "Under 6-month PM, each cycle's effective exposure is ~6 months before intervention resets wear accumulation. Expected failures = lambda * 6.",
      "data_source": "derived",
      "depends_on": ["posterior_failure_rate"]
    },
    {
      "name": "failures_next_12mo_rtf",
      "description": "Predicted failures in next 12 months under run-to-failure",
      "role": "decision_variable",
      "distribution_family": "Poisson",
      "distribution_parameters": {"mu": "posterior_failure_rate * 12"},
      "justification": "Full 12-month exposure with no preventive interventions.",
      "data_source": "derived",
      "depends_on": ["posterior_failure_rate"]
    },
    {
      "name": "total_cost_pm",
      "description": "Total cost under PM: 2 planned interventions + unplanned failures",
      "role": "cost_node",
      "distribution_family": "derived_from_simulation",
      "distribution_parameters": {"pm_interventions": 2, "cost_pm": 4200, "cost_failure": 18000},
      "justification": "2 planned PMs per year × $4,200 + expected unplanned failures × $18,000.",
      "data_source": "derived",
      "depends_on": ["failures_next_12mo_pm"]
    },
    {
      "name": "total_cost_rtf",
      "description": "Total cost under run-to-failure: all costs are unplanned failure costs",
      "role": "cost_node",
      "distribution_family": "derived_from_simulation",
      "distribution_parameters": {"cost_failure": 18000},
      "justification": "No planned interventions; all maintenance costs come from unplanned failures.",
      "data_source": "derived",
      "depends_on": ["failures_next_12mo_rtf"]
    }
  ],
  "bayesian_update_chain": [
    "failure_rate_lambda",
    "observed_failures",
    "posterior_failure_rate",
    "failures_next_12mo_pm",
    "failures_next_12mo_rtf",
    "total_cost_pm",
    "total_cost_rtf"
  ],
  "voi_candidates": ["failure_rate_lambda", "posterior_failure_rate"],
  "model_notes": "Gamma-Poisson conjugate model allows analytical posterior. Monte Carlo used for cost propagation. PM effectiveness assumed to reset wear accumulation at each intervention."
}

# Uncomment the line below to use offline fallback:
# bayesian_model_json = OFFLINE_JSON

print("Offline fallback JSON defined. Uncomment last line to use it.")

---
## Section 5 — Step 3: Value of Information (VoI) Ranking

Before collecting more data, AIE asks: **which uncertainties are actually worth resolving?**

We calculate **Expected Value of Perfect Information (EVPI)** for each measurement node.  
EVPI tells us the maximum we should spend to perfectly learn a variable's value.

$$\text{EVPI} = E[\text{Cost under uncertainty}] - E[\text{Cost with perfect information}]$$

Nodes with **EVPI < measurement cost** are not worth measuring further.

In [ ]:
# ── VoI ANALYSIS ──────────────────────────────────────────────────────────────
# We simulate EVPI by comparing expected cost under current uncertainty
# versus expected cost if we knew the failure rate exactly.

N_SIM = 50_000  # Monte Carlo samples
rng   = np.random.default_rng(seed=42)

# Posterior Gamma parameters (from conjugate update)
alpha_post = 3.5 + 3   # prior_alpha + observed_failures
beta_post  = 12.0 + 24 # prior_beta  + observation_period

# Draw samples of failure rate from posterior
lambda_samples = rng.gamma(shape=alpha_post, scale=1/beta_post, size=N_SIM)

# Cost parameters
C_PM      = decision_frame["cost_pm_per_intervention_usd"]
C_FAIL    = decision_frame["cost_unplanned_failure_usd"]
HORIZON   = decision_frame["time_horizon_months"]
PM_CYCLE  = 6   # months between PM interventions
N_PM      = HORIZON // PM_CYCLE  # number of planned PMs per year

# Expected failures per strategy
exp_fail_rtf = lambda_samples * HORIZON
exp_fail_pm  = lambda_samples * PM_CYCLE   # failures per PM cycle × 2 cycles

# Total cost per strategy (Monte Carlo)
cost_rtf = rng.poisson(exp_fail_rtf) * C_FAIL
cost_pm  = (N_PM * C_PM) + rng.poisson(exp_fail_pm * N_PM) * C_FAIL

# Decision under uncertainty: choose the strategy with lower expected cost
best_strategy_cost = np.minimum(cost_rtf, cost_pm)
E_cost_uncertainty = np.mean(np.minimum(cost_rtf, cost_pm))

# EVPI: if we knew lambda exactly, we'd always pick the right strategy
# For each simulated lambda, pick the cheaper option
cost_rtf_det = lambda_samples * HORIZON * C_FAIL
cost_pm_det  = N_PM * C_PM + lambda_samples * PM_CYCLE * N_PM * C_FAIL
E_cost_perfect = np.mean(np.minimum(cost_rtf_det, cost_pm_det))

EVPI = E_cost_uncertainty - E_cost_perfect

print(f"Expected cost (uncertainty, best strategy):  ${E_cost_uncertainty:,.0f}")
print(f"Expected cost (perfect information):         ${E_cost_perfect:,.0f}")
print(f"EVPI (max value of learning true lambda):    ${EVPI:,.0f}")
print()
print("Interpretation:")
if EVPI > 500:
    print(f"  → It is worth spending up to ${EVPI:,.0f} to better characterize the failure rate.")
    print("  → Consider: vibration monitoring, oil analysis, or additional run time before deciding.")
else:
    print("  → EVPI is low. The current data is sufficient to make the decision. Proceed to recommendation.")

In [ ]:
# ── VoI BAR CHART ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cost distribution comparison
axes[0].hist(cost_rtf / 1000, bins=60, alpha=0.6, label="Run-to-Failure", color="#e74c3c")
axes[0].hist(cost_pm  / 1000, bins=60, alpha=0.6, label="6-Month PM",     color="#2ecc71")
axes[0].axvline(np.mean(cost_rtf)/1000, color="#c0392b", lw=2, linestyle="--",
                label=f"RTF mean: ${np.mean(cost_rtf)/1000:.1f}k")
axes[0].axvline(np.mean(cost_pm)/1000,  color="#27ae60", lw=2, linestyle="--",
                label=f"PM mean:  ${np.mean(cost_pm)/1000:.1f}k")
axes[0].set_xlabel("Total 12-Month Cost ($k)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Cost Distribution by Strategy")
axes[0].legend(fontsize=8)

# VoI summary bar
voi_items = {
    "Failure Rate (λ)": EVPI,
    "PM Effectiveness": EVPI * 0.35,   # Illustrative relative values
    "Failure Cost Estimate": EVPI * 0.15,
}
axes[1].barh(list(voi_items.keys()), [v/1000 for v in voi_items.values()],
             color=["#3498db", "#9b59b6", "#f39c12"])
axes[1].set_xlabel("EVPI ($k)")
axes[1].set_title("Value of Information by Measurement")
axes[1].axvline(0.5, color="gray", linestyle=":", label="$500 measurement cost")
axes[1].legend(fontsize=8)

plt.suptitle("AIE Step 3 — Value of Information Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 6 — Step 4: Calibrated Priors (Two Paths)

AIE requires **calibrated probability estimates** — not guesses, not overconfident point values.  
This section shows both paths for seeding priors:

### Path A — Expert Estimate → 90% Confidence Interval → Distribution
Engineer provides a 90% CI ("I'm 90% confident the MTBF is between 8 and 16 months").  
We fit a distribution to that interval.

### Path B — Sample Data → MLE / Method of Moments → Distribution
We have observed failure times. We fit a distribution directly and test goodness-of-fit.

In [ ]:
# ── PATH A: Expert Calibrated Estimate ─────────────────────────────────────────
# Engineer's 90% CI for MTBF: [8, 16] months
# → Convert to failure rate lambda = 1/MTBF
# → lambda CI: [1/16, 1/8] = [0.0625, 0.125] failures/month

mtbf_low_90  = 8    # months (5th percentile estimate)
mtbf_high_90 = 16   # months (95th percentile estimate)

lambda_low  = 1 / mtbf_high_90   # 0.0625
lambda_high = 1 / mtbf_low_90    # 0.125

# Fit Gamma distribution parameters to match the 90% CI
# Using method: find (alpha, beta) such that Gamma CDF at low = 0.05, high = 0.95
from scipy.optimize import minimize

def gamma_ci_loss(params):
    alpha, rate = params
    if alpha <= 0 or rate <= 0:
        return 1e10
    p5  = stats.gamma.ppf(0.05, a=alpha, scale=1/rate)
    p95 = stats.gamma.ppf(0.95, a=alpha, scale=1/rate)
    return (p5 - lambda_low)**2 + (p95 - lambda_high)**2

result = minimize(gamma_ci_loss, x0=[3.5, 35.0], method="Nelder-Mead")
alpha_expert, rate_expert = result.x
mean_expert = alpha_expert / rate_expert

print("PATH A — Expert Calibrated Prior:")
print(f"  MTBF 90% CI entered: [{mtbf_low_90}, {mtbf_high_90}] months")
print(f"  Lambda 90% CI:       [{lambda_low:.4f}, {lambda_high:.4f}] failures/month")
print(f"  Fitted Gamma:        alpha={alpha_expert:.2f}, rate={rate_expert:.2f}")
print(f"  Prior mean lambda:   {mean_expert:.4f} → implied MTBF = {1/mean_expert:.1f} months")

In [ ]:
# ── PATH B: Sample Data → Fitted Distribution ──────────────────────────────────
# Simulated historical inter-failure times (months) for demonstration.
# Replace with your actual data.

observed_times_to_failure = np.array([8.2, 11.5, 9.8, 14.1, 10.3, 7.6, 12.9, 8.8, 11.1, 9.4])

# Fit Exponential (1-parameter) and Weibull (2-parameter) — compare
exp_params    = stats.expon.fit(observed_times_to_failure, floc=0)
weibull_params = stats.weibull_min.fit(observed_times_to_failure, floc=0)

# Log-likelihood comparison (higher = better fit)
ll_exp     = np.sum(stats.expon.logpdf(observed_times_to_failure, *exp_params))
ll_weibull = np.sum(stats.weibull_min.logpdf(observed_times_to_failure, *weibull_params))

print("PATH B — Data-Fitted Prior:")
print(f"  Sample size: {len(observed_times_to_failure)} observed TTFs")
print(f"  Sample mean: {np.mean(observed_times_to_failure):.2f} months")
print()
print(f"  Exponential fit:  scale={exp_params[1]:.2f}  | log-likelihood={ll_exp:.2f}")
print(f"  Weibull fit:      shape={weibull_params[0]:.2f}, scale={weibull_params[2]:.2f} | log-likelihood={ll_weibull:.2f}")
print()
better = "Weibull" if ll_weibull > ll_exp else "Exponential"
print(f"  → {better} provides better fit (higher log-likelihood).")

# Weibull shape interpretation
k = weibull_params[0]
if k < 1:
    wear = "infant mortality / early-life failures (decreasing hazard)"
elif k == 1:
    wear = "random failures (constant hazard — memoryless)"
else:
    wear = "wear-out failures (increasing hazard — PM is beneficial)"
print(f"  Weibull shape k={k:.2f} → {wear}")

In [ ]:
# ── PRIOR COMPARISON PLOT ──────────────────────────────────────────────────────
x = np.linspace(0.01, 0.25, 500)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Path A: Expert Gamma prior on lambda
prior_pdf = stats.gamma.pdf(x, a=alpha_expert, scale=1/rate_expert)
post_pdf  = stats.gamma.pdf(x, a=alpha_post,   scale=1/beta_post)
axes[0].plot(x, prior_pdf, "b--", lw=2, label=f"Expert Prior Gamma({alpha_expert:.1f}, {rate_expert:.1f})")
axes[0].plot(x, post_pdf,  "g-",  lw=2.5, label=f"Posterior Gamma({alpha_post:.1f}, {beta_post:.1f})")
axes[0].fill_between(x, post_pdf, alpha=0.15, color="green")
axes[0].axvline(alpha_post/beta_post, color="green", lw=1.5, linestyle=":",
                label=f"Posterior mean λ={alpha_post/beta_post:.3f}")
axes[0].set_xlabel("Failure Rate λ (failures/month)")
axes[0].set_ylabel("Probability Density")
axes[0].set_title("Path A — Expert Prior → Posterior Update")
axes[0].legend(fontsize=8)

# Path B: Weibull fit on TTF data
t = np.linspace(0, 25, 500)
weibull_pdf = stats.weibull_min.pdf(t, *weibull_params)
exp_pdf     = stats.expon.pdf(t, *exp_params)
axes[1].hist(observed_times_to_failure, bins=8, density=True, alpha=0.4,
             color="steelblue", label="Observed TTF data")
axes[1].plot(t, weibull_pdf, "r-",  lw=2, label=f"Weibull (k={weibull_params[0]:.2f})")
axes[1].plot(t, exp_pdf,     "k--", lw=2, label="Exponential")
axes[1].set_xlabel("Time to Failure (months)")
axes[1].set_ylabel("Probability Density")
axes[1].set_title("Path B — Data-Fitted Distribution")
axes[1].legend(fontsize=8)

plt.suptitle("AIE Step 4 — Calibrated Priors: Two Paths", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 7 — Step 5: Bayesian Update (PyMC Model)

We now build the full probabilistic model in **PyMC**, which:
- Encodes the prior on failure rate
- Conditions on observed failure count (likelihood)
- Samples the posterior via MCMC (NUTS sampler)

This is equivalent to the analytical Gamma-Poisson conjugate result, but generalizable to any prior/likelihood combination.

$$\underbrace{P(\lambda \mid k, t)}_{\text{posterior}} \propto \underbrace{\text{Poisson}(k \mid \lambda t)}_{\text{likelihood}} \cdot \underbrace{\text{Gamma}(\lambda \mid \alpha, \beta)}_{\text{prior}}$$

In [ ]:
# ── PyMC BAYESIAN MODEL ────────────────────────────────────────────────────────
# Prior: Gamma(alpha, beta) on failure rate
# Likelihood: Poisson(lambda * t) observed k failures

obs_failures = decision_frame["observed_failures"]
obs_months   = decision_frame["observation_period_months"]

with pm.Model() as reliability_model:

    # Prior on failure rate (from expert calibration — Path A)
    lambda_rate = pm.Gamma(
        "lambda_rate",
        alpha=alpha_expert,
        beta=rate_expert,
        initval=mean_expert
    )

    # Likelihood: observed failure count over observation period
    failures_observed = pm.Poisson(
        "failures_observed",
        mu=lambda_rate * obs_months,
        observed=obs_failures
    )

    # Posterior predictive: expected failures under each strategy
    fail_rtf = pm.Poisson("fail_rtf", mu=lambda_rate * HORIZON)
    fail_pm  = pm.Poisson("fail_pm",  mu=lambda_rate * PM_CYCLE * N_PM)

    # Sample posterior
    trace = pm.sample(
        draws=2000,
        tune=1000,
        target_accept=0.9,
        progressbar=True,
        return_inferencedata=True,
        random_seed=42
    )

print("\nBayesian sampling complete.")

In [ ]:
# ── POSTERIOR SUMMARY ─────────────────────────────────────────────────────────
summary = az.summary(trace, var_names=["lambda_rate"], round_to=4)
display(summary)

# Compare analytical vs MCMC posterior
lambda_post_mean_analytical = alpha_post / beta_post
lambda_post_mean_mcmc       = float(trace.posterior["lambda_rate"].mean())

print(f"\nAnalytical posterior mean λ: {lambda_post_mean_analytical:.4f} → MTBF = {1/lambda_post_mean_analytical:.1f} months")
print(f"MCMC posterior mean λ:       {lambda_post_mean_mcmc:.4f} → MTBF = {1/lambda_post_mean_mcmc:.1f} months")
print(f"Difference:                  {abs(lambda_post_mean_analytical - lambda_post_mean_mcmc):.5f} (should be ~0 for conjugate model)")

# Posterior plot
az.plot_posterior(trace, var_names=["lambda_rate"],
                  hdi_prob=0.9,
                  point_estimate="mean",
                  ref_val=mean_expert)
plt.suptitle("Posterior Distribution of Failure Rate λ\n(ref_val = expert prior mean)",
             fontsize=11)
plt.tight_layout()
plt.show()

---
## Section 8 — Step 5b: Chained Bayesian DAG Visualization

The JSON from Section 4 defines a **Directed Acyclic Graph (DAG)** — a chain of dependencies  
from raw measurements → derived quantities → cost outputs.

This visualization shows exactly **where each piece of information enters the model**.

In [ ]:
# ── DAG VISUALIZATION FROM JSON MODEL ─────────────────────────────────────────
# Build NetworkX graph from the LLM-generated JSON

model_json = bayesian_model_json if "bayesian_model_json" in dir() else OFFLINE_JSON

G = nx.DiGraph()

# Color coding by node role
role_colors = {
    "prior":            "#3498db",   # blue
    "likelihood":       "#e67e22",   # orange
    "posterior":        "#2ecc71",   # green
    "decision_variable":"#9b59b6",   # purple
    "cost_node":        "#e74c3c",   # red
}

nodes      = model_json["measurement_nodes"]
node_names = {n["name"] for n in nodes}

for node in nodes:
    G.add_node(node["name"],
               role=node["role"],
               label=node["name"].replace("_", "\n"))

for node in nodes:
    for dep in node.get("depends_on", []):
        if dep in node_names:
            G.add_edge(dep, node["name"])

# Layout and draw
pos    = nx.nx_agraph.graphviz_layout(G, prog="dot") if nx.nx_agraph else nx.spring_layout(G, seed=1)
colors = [role_colors.get(G.nodes[n]["role"], "#95a5a6") for n in G.nodes]
labels = {n: G.nodes[n]["label"] for n in G.nodes}

fig, ax = plt.subplots(figsize=(14, 6))
nx.draw_networkx(
    G, pos=pos, ax=ax,
    labels=labels,
    node_color=colors,
    node_size=2200,
    font_size=7,
    font_color="white",
    font_weight="bold",
    edge_color="#555",
    arrows=True,
    arrowsize=18,
    width=2,
)

# Legend
legend_patches = [mpatches.Patch(color=v, label=k.replace("_", " ").title())
                  for k, v in role_colors.items()]
ax.legend(handles=legend_patches, loc="upper left", fontsize=8)
ax.set_title("Bayesian Measurement DAG — Generated from Problem Description",
             fontsize=12, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

# Print chain
chain = model_json["bayesian_update_chain"]
display(Markdown("**Bayesian Update Chain:** " + " → ".join(f"`{n}`" for n in chain)))

---
## Section 9 — Step 7: Monte Carlo Simulation & Decision Recommendation

Propagate posterior uncertainty through the cost model to produce a **full cost distribution**  
for each strategy, and generate an **auditable decision recommendation**.

In [ ]:
# ── MONTE CARLO COST PROPAGATION ───────────────────────────────────────────────
# Draw lambda samples from PyMC posterior trace

lambda_posterior_samples = trace.posterior["lambda_rate"].values.flatten()
N = len(lambda_posterior_samples)
rng2 = np.random.default_rng(seed=99)

# Simulate failure counts under each strategy
k_rtf  = rng2.poisson(lambda_posterior_samples * HORIZON)
k_pm   = rng2.poisson(lambda_posterior_samples * PM_CYCLE * N_PM)

# Total cost
total_cost_rtf = k_rtf * C_FAIL
total_cost_pm  = (N_PM * C_PM) + k_pm * C_FAIL

# Summary statistics
results = pd.DataFrame({
    "Strategy":     ["Run-to-Failure", "6-Month PM"],
    "Mean Cost ($)": [np.mean(total_cost_rtf), np.mean(total_cost_pm)],
    "Median ($)":   [np.median(total_cost_rtf), np.median(total_cost_pm)],
    "P10 ($)":      [np.percentile(total_cost_rtf, 10), np.percentile(total_cost_pm, 10)],
    "P90 ($)":      [np.percentile(total_cost_rtf, 90), np.percentile(total_cost_pm, 90)],
    "P(cheaper) %": [
        np.mean(total_cost_rtf < total_cost_pm) * 100,
        np.mean(total_cost_pm  < total_cost_rtf) * 100
    ]
})
results = results.set_index("Strategy")
results = results.applymap(lambda x: f"${x:,.0f}" if "$" in str(x) or x > 10 else f"{x:.1f}%"
                           if isinstance(x, float) else x)

display(Markdown("### Monte Carlo Cost Summary"))
display(results)

In [ ]:
# ── DECISION VISUALIZATION ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cost CDFs
for cost, label, color in [
    (total_cost_rtf, "Run-to-Failure", "#e74c3c"),
    (total_cost_pm,  "6-Month PM",     "#2ecc71")
]:
    sorted_cost = np.sort(cost)
    cdf = np.arange(1, len(sorted_cost)+1) / len(sorted_cost)
    axes[0].plot(sorted_cost/1000, cdf, lw=2.5, label=label, color=color)

axes[0].set_xlabel("Total 12-Month Cost ($k)")
axes[0].set_ylabel("Cumulative Probability")
axes[0].set_title("Cost CDF by Strategy")
axes[0].axhline(0.5, color="gray", linestyle=":", lw=1)
axes[0].legend()

# Savings distribution
savings = total_cost_rtf - total_cost_pm
axes[1].hist(savings/1000, bins=50, color="#3498db", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="black", lw=2, linestyle="--", label="Break-even")
axes[1].axvline(np.mean(savings)/1000, color="#e74c3c", lw=2,
                label=f"Mean savings: ${np.mean(savings)/1000:.1f}k")
pct_pm_wins = np.mean(savings > 0) * 100
axes[1].set_xlabel("Cost Savings from PM vs RTF ($k)")
axes[1].set_ylabel("Frequency")
axes[1].set_title(f"PM Cheaper in {pct_pm_wins:.0f}% of Simulations")
axes[1].legend(fontsize=9)

plt.suptitle("Monte Carlo Decision Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ── RECOMMENDATION ────────────────────────────────────────────────────────────
mean_savings = np.mean(savings)
rec = "6-Month Preventive Maintenance" if mean_savings > 0 else "Run-to-Failure"
display(Markdown(f"""
---
### Decision Recommendation

**Recommended Strategy**: **{rec}**

- PM is the lower-cost strategy in **{pct_pm_wins:.0f}%** of simulated scenarios.
- Expected savings from PM over 12 months: **${mean_savings:,.0f}**
- EVPI = **${EVPI:,.0f}** — additional investment in failure-rate characterization  
  (e.g., vibration monitoring) is {'justified' if EVPI > 500 else 'not required'}.

*This recommendation is based on {N:,} posterior draws and {N:,} Monte Carlo simulations.*
"""))

---
## Section 10 — Your Turn: Participant Exercise

Replace the decision frame below with **your own reliability problem**.  
The LLM will map it into a Bayesian structure automatically.

**Suggested exercise scenarios:**
1. A conveyor belt system with 5 failures over 18 months — decide on predictive vs. scheduled maintenance.
2. A safety valve with no failures in 36 months — is the current inspection interval adequate?
3. A fleet of 12 motors with mixed failure history — prioritize which unit to maintain first.

---
### ## YOUR TURN — Edit the cell below

In [ ]:
# ── PARTICIPANT EXERCISE ───────────────────────────────────────────────────────
# Replace the values below with your own problem.
# Then re-run ALL cells from Section 3 onward.

my_decision_frame = {
    "problem_description": """
        << DESCRIBE YOUR RELIABILITY PROBLEM HERE >>
        Include: asset type, failure history, costs, time horizon, and decision options.
    """,
    "decision_options": [
        "Option A",
        "Option B"
    ],
    "objective": "Minimize expected cost over the planning horizon",
    "time_horizon_months": 12,
    "cost_pm_per_intervention_usd": 0,     # Replace with your PM cost
    "cost_unplanned_failure_usd": 0,       # Replace with your failure cost
    "observed_failures": 0,                # Replace with observed failure count
    "observation_period_months": 12,       # Replace with observation duration
    "industry_mtbf_range_months": [6, 18], # Replace with your 90% CI on MTBF
}

# Uncomment to use your frame and re-run from Section 3:
# decision_frame = my_decision_frame
# bayesian_model_json = parse_problem_to_json(my_decision_frame["problem_description"])
# print(json.dumps(bayesian_model_json, indent=2))

print("Exercise template ready. Fill in the fields above and uncomment the last lines.")

---
## References

1. **Hubbard, D.** (2014). *How to Measure Anything: Finding the Value of Intangibles in Business* (3rd ed.). Wiley.
2. **Hubbard, D.** (2025). *How to Measure Anything in AI: Quantitative Techniques for Decision-Making*. LinkedIn Learning.
3. **LLM-BI**: *Towards Fully Automated Bayesian Inference with Large Language Models*. arXiv:2508.08300 (Aug 2025).
4. **Lorenz & Fritz** (2026). *Scalable Delphi: LLM-Assisted Expert Elicitation for Calibrated Priors*. arXiv (Feb 2026).
5. **Salvatier, J., Wiecki, T., Fonnesbeck, C.** (2016). Probabilistic programming in Python using PyMC3. *PeerJ Computer Science*.
6. **Kumar & Klefsjö** (1994). Proportional hazards model: a review. *Reliability Engineering & System Safety*, 44(2), 177-188.

---
*Notebook version 1.0 — RAMS 2027 Tutorial Submission*  
*Author: Kirtis Christensen | Bayesian Mentor: Grok (xAI)*  
*Generated: April 30, 2026*